## Used Bike Market Analysis using Web Scraping

**Website:**  https://bikekharido.in/

**Objective:** To scrape used-bike listings from BikeKharido and analyze bike prices, brands, mileage driven, ownership and locations to identify patterns in the used-bike market.

**Targeted Columns:** 
   - bike_name
   - brand
   - price
   - km_driven
   - ownership
   - location
   - year

The website currently exposes these kinds of fields in its listings.

**Final workflow:**
Website Selection

       ↓
Data Collection / Web Scraping

       ↓
Data Understanding

       ↓
Data Cleaning

       ↓   
EDA

       ↓
Data Visualization

       ↓
Business Insights

       ↓
Recommendations

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [2]:
url = "https://bikekharido.in/used-bikes-in-india/"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

print(response.status_code)
print(len(response.text))

200
817216


In [3]:
soup = BeautifulSoup(response.text, "html.parser")

print(soup.title.text)

Buy Second Hand Bike in India | Buy & Sell Used Bikes - BikeKharido


In [4]:
listing = soup.find(string=lambda text: text and "Seller Demand" in text)

print(listing)

Seller Demand 


In [5]:
parent = listing.parent

print(parent)

<span class="boxlabel">Seller Demand </span>


In [6]:
parent = listing.parent

print(parent)

<span class="boxlabel">Seller Demand </span>


In [7]:
container = listing.parent.parent

print(container.get_text(" ", strip=True))

Seller Demand ₹ 25,000


In [8]:
print(container.parent.prettify()[:5000])

<div class="price_heading">
 <div class="ex-price">
  <!-- <p></p> -->
  <span class="boxlabel">
   Seller Demand
  </span>
  <span class="fw500 spanblock fsize17 fcblack">
   ₹ 25,000
  </span>
 </div>
</div>



In [9]:
card = container

for i in range(5):
    card = card.parent
    print("LEVEL", i + 1)
    print("CLASS:", card.get("class"))
    print("TEXT:", card.get_text(" ", strip=True)[:500])
    print("-" * 80)

LEVEL 1
CLASS: ['price_heading']
TEXT: Seller Demand ₹ 25,000
--------------------------------------------------------------------------------
LEVEL 2
CLASS: ['price_ribbon_disable']
TEXT: Seller Demand ₹ 25,000
--------------------------------------------------------------------------------
LEVEL 3
CLASS: ['fl-post-text']
TEXT: Activa 3G STD By Shubham Jain | September 18, 2026 Seller Demand ₹ 25,000 KM Driven 60000 km Ownership First Location Mumbai
--------------------------------------------------------------------------------
LEVEL 4
CLASS: None
TEXT: +4 photos Activa 3G STD By Shubham Jain | September 18, 2026 Seller Demand ₹ 25,000 KM Driven 60000 km Ownership First Location Mumbai
--------------------------------------------------------------------------------
LEVEL 5
CLASS: None
TEXT: +4 photos Activa 3G STD By Shubham Jain | September 18, 2026 Seller Demand ₹ 25,000 KM Driven 60000 km Ownership First Location Mumbai
------------------------------------------------------------

In [10]:
cards = soup.find_all("div", class_="fl-post-text")

print("Number of listings found:", len(cards))

for card in cards[:5]:
    print(card.get_text(" ", strip=True))
    print("-" * 80)

Number of listings found: 24
Activa 3G STD By Shubham Jain | September 18, 2026 Seller Demand ₹ 25,000 KM Driven 60000 km Ownership First Location Mumbai
--------------------------------------------------------------------------------
Maestro Edge 110 Drum Brake Alloy Wheel FI By Shubham Jain | September 17, 2026 Seller Demand ₹ 34,000 KM Driven 28000 km Ownership Third Location Jaipur
--------------------------------------------------------------------------------
XPulse 200 4V By Shubham Jain | September 16, 2026 Seller Demand ₹ 95,000 KM Driven 25000 km Ownership Second Location Mumbai
--------------------------------------------------------------------------------
S1 X By Shubham Jain | August 10, 2026 Seller Demand ₹ 75,000 KM Driven 2250 km Ownership First Location Kolkata
--------------------------------------------------------------------------------
Karizma R By Shubham Jain | August 10, 2026 Seller Demand ₹ 16,000 KM Driven 100000 km Ownership First Location Hyderabad
-------

In [11]:
card = cards[0]

print(card.find("h3").get_text(strip=True) if card.find("h3") else "No h3 found")

No h3 found


In [12]:
print(card.prettify()[:5000])

<div class="fl-post-text">
 <h2 class="fl-post-title-old bikeListHead">
  <a href="https://bikekharido.in/buy-used-bike/activa-3g-std-2/" title="Activa 3G STD">
   Activa 3G STD
  </a>
 </h2>
 <div class="fl-post-meta hide">
  By
  <a href="https://bikekharido.in/author/admin/">
   Shubham Jain
  </a>
  <span class="fl-post-meta-sep">
   |
  </span>
  September 18, 2026
 </div>
 <div class="price_ribbon_disable">
  <div class="price_heading">
   <div class="ex-price">
    <!-- <p></p> -->
    <span class="boxlabel">
     Seller Demand
    </span>
    <span class="fw500 spanblock fsize17 fcblack">
     ₹ 25,000
    </span>
   </div>
  </div>
 </div>
 <div class="otherDetails">
  <div class="vehicleMetaDetails">
   <span class="labelkey">
    KM Driven
   </span>
   <span class="boxlabel imptags">
    60000 km
   </span>
  </div>
  <div class="vehicleMetaDetails">
   <span class="labelkey">
    Ownership
   </span>
   <span class="boxlabel imptags">
    First
   </span>
  </div>
  <div c

In [13]:
card = cards[0]

bike_name = card.find("h2", class_="bikeListHead").get_text(strip=True)

price = card.find("div", class_="price_heading").find_all("span")[1].get_text(strip=True)

details = card.find_all("div", class_="vehicleMetaDetails")

km_driven = details[0].find("span", class_="boxlabel").get_text(strip=True)
ownership = details[1].find("span", class_="boxlabel").get_text(strip=True)
location = details[2].find("span", class_="boxlabel").get_text(strip=True)

print("Bike Name:", bike_name)
print("Price:", price)
print("KM Driven:", km_driven)
print("Ownership:", ownership)
print("Location:", location)

Bike Name: Activa 3G STD
Price: ₹ 25,000
KM Driven: 60000 km
Ownership: First
Location: Mumbai


In [14]:
bike_data = []

for card in cards:
    bike_name = card.find("h2", class_="bikeListHead").get_text(strip=True)

    price = card.find("div", class_="price_heading").find_all("span")[1].get_text(strip=True)

    details = card.find_all("div", class_="vehicleMetaDetails")

    km_driven = details[0].find("span", class_="boxlabel").get_text(strip=True)
    ownership = details[1].find("span", class_="boxlabel").get_text(strip=True)
    location = details[2].find("span", class_="boxlabel").get_text(strip=True)

    bike_data.append({
        "bike_name": bike_name,
        "price": price,
        "km_driven": km_driven,
        "ownership": ownership,
        "location": location
    })

df = pd.DataFrame(bike_data)

print("Number of records:", len(df))
df.head()

Number of records: 24


,bike_name,price,km_driven,ownership,location
0,Activa 3G STD,"₹ 25,000",60000 km,First,Mumbai
1,Maestro Edge 110 Drum Brake Alloy Wheel FI,"₹ 34,000",28000 km,Third,Jaipur
2,XPulse 200 4V,"₹ 95,000",25000 km,Second,Mumbai
3,S1 X,"₹ 75,000",2250 km,First,Kolkata
4,Karizma R,"₹ 16,000",100000 km,First,Hyderabad


In [15]:
for a in soup.find_all("a", href=True):
    text = a.get_text(" ", strip=True)
    href = a.get("href")

    if text.lower() in ["next", "next page"] or "page/" in href.lower():
        print(text, href)

In [16]:
df.to_csv("bikekharido_used_bikes_raw.csv", index=False)

print("Raw data saved successfully")

Raw data saved successfully


In [17]:
# check all links on the page

links = soup.find_all("a", href=True)

for link in links:
    href = link["href"]
    
    if "page" in href.lower():
        print(href)

https://bikekharido.in/used-bikes-in-india/?sf_paged=2
https://bikekharido.in/used-bikes-in-india/?sf_paged=3
https://bikekharido.in/used-bikes-in-india/?sf_paged=63
https://bikekharido.in/used-bikes-in-india/?sf_paged=2
https://bikekharido.in/used-bikes-in-india/?sf_paged=2
https://bikekharido.in/used-bikes-in-india/?sf_paged=3
https://bikekharido.in/used-bikes-in-india/?sf_paged=63
https://bikekharido.in/used-bikes-in-india/?sf_paged=2


In [18]:
url_page2 = "https://bikekharido.in/used-bikes-in-india/?sf_paged=2"

response_page2 = requests.get(url_page2, headers=headers)

print(response_page2.status_code)
print(len(response_page2.text))

200
818901


In [19]:
soup_page2 = BeautifulSoup(response_page2.text, "html.parser")

cards_page2 = soup_page2.find_all("div", class_="fl-post-text")

print("Number of listings on page 2:", len(cards_page2))

Number of listings on page 2: 24


In [20]:
import time

all_bikes = []

for page in range(1, 26):
    
    if page == 1:
        url = "https://bikekharido.in/used-bikes-in-india/"
    else:
        url = f"https://bikekharido.in/used-bikes-in-india/?sf_paged={page}"
    
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    
    cards = soup.find_all("div", class_="fl-post-text")
    
    print(f"Page {page}: {len(cards)} listings")
    
    for card in cards:
        try:
            bike_name = card.find(
                "h2", class_="bikeListHead"
            ).get_text(strip=True)
            
            price = card.find(
                "div", class_="price_heading"
            ).find_all("span")[1].get_text(strip=True)
            
            details = card.find_all(
                "div", class_="vehicleMetaDetails"
            )
            
            km_driven = details[0].find(
                "span", class_="boxlabel"
            ).get_text(strip=True)
            
            ownership = details[1].find(
                "span", class_="boxlabel"
            ).get_text(strip=True)
            
            location = details[2].find(
                "span", class_="boxlabel"
            ).get_text(strip=True)
            
            all_bikes.append({
                "bike_name": bike_name,
                "price": price,
                "km_driven": km_driven,
                "ownership": ownership,
                "location": location
            })
            
        except Exception:
            continue
    
    time.sleep(1)

df_raw = pd.DataFrame(all_bikes)

print("\nTotal scraped rows:", len(df_raw))

Page 1: 24 listings
Page 2: 24 listings
Page 3: 24 listings
Page 4: 24 listings
Page 5: 24 listings
Page 6: 24 listings
Page 7: 24 listings
Page 8: 24 listings
Page 9: 24 listings
Page 10: 24 listings
Page 11: 24 listings
Page 12: 24 listings
Page 13: 24 listings
Page 14: 24 listings
Page 15: 24 listings
Page 16: 24 listings
Page 17: 24 listings
Page 18: 24 listings
Page 19: 24 listings
Page 20: 24 listings
Page 21: 24 listings
Page 22: 24 listings
Page 23: 24 listings
Page 24: 24 listings
Page 25: 24 listings

Total scraped rows: 600


In [21]:
print(df_raw.shape)

(600, 5)


In [22]:
df_raw.head()

,bike_name,price,km_driven,ownership,location
0,Activa 3G STD,"₹ 25,000",60000 km,First,Mumbai
1,Maestro Edge 110 Drum Brake Alloy Wheel FI,"₹ 34,000",28000 km,Third,Jaipur
2,XPulse 200 4V,"₹ 95,000",25000 km,Second,Mumbai
3,S1 X,"₹ 75,000",2250 km,First,Kolkata
4,Karizma R,"₹ 16,000",100000 km,First,Hyderabad


In [23]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   bike_name  600 non-null    object
 1   price      600 non-null    object
 2   km_driven  600 non-null    object
 3   ownership  600 non-null    object
 4   location   600 non-null    object
dtypes: object(5)
memory usage: 23.6+ KB


In [24]:
df_raw.to_csv(
    "bikekharido_used_bikes_collected.csv",
    index=False
)

print("Raw dataset saved successfully!")

Raw dataset saved successfully!


In [25]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# Website
base_url = "https://bikekharido.in/used-bikes-in-india/"

# Request headers
headers = {
    "User-Agent": "Mozilla/5.0"
}

# Empty list to store all listings
all_bikes = []

# Scrape pages 1 to 50
for page in range(1, 51):

    # Page 1 has a different URL
    if page == 1:
        url = base_url
    else:
        url = f"{base_url}?sf_paged={page}"

    response = requests.get(url, headers=headers)

    soup = BeautifulSoup(response.text, "html.parser")

    cards = soup.find_all("div", class_="fl-post-text")

    print(f"Page {page}: {len(cards)} listings")

    for card in cards:

        try:
            bike_name = card.find(
                "h2", class_="bikeListHead"
            ).get_text(strip=True)

            price = card.find(
                "div", class_="price_heading"
            ).find_all("span")[1].get_text(strip=True)

            details = card.find_all(
                "div", class_="vehicleMetaDetails"
            )

            km_driven = details[0].find(
                "span", class_="boxlabel"
            ).get_text(strip=True)

            ownership = details[1].find(
                "span", class_="boxlabel"
            ).get_text(strip=True)

            location = details[2].find(
                "span", class_="boxlabel"
            ).get_text(strip=True)

            all_bikes.append({
                "bike_name": bike_name,
                "price": price,
                "km_driven": km_driven,
                "ownership": ownership,
                "location": location
            })

        except Exception:
            continue

    # Small delay between requests
    time.sleep(1)

# Create DataFrame
df_raw = pd.DataFrame(all_bikes)

# Display result
print("\nScraping completed!")
print("Total raw rows:", len(df_raw))
print("Columns:", list(df_raw.columns))

Page 1: 24 listings
Page 2: 24 listings
Page 3: 24 listings
Page 4: 24 listings
Page 5: 24 listings
Page 6: 24 listings
Page 7: 24 listings
Page 8: 24 listings
Page 9: 24 listings
Page 10: 24 listings
Page 11: 24 listings
Page 12: 24 listings
Page 13: 24 listings
Page 14: 24 listings
Page 15: 24 listings
Page 16: 24 listings
Page 17: 24 listings
Page 18: 24 listings
Page 19: 24 listings
Page 20: 24 listings
Page 21: 24 listings
Page 22: 24 listings
Page 23: 24 listings
Page 24: 24 listings
Page 25: 24 listings
Page 26: 24 listings
Page 27: 24 listings
Page 28: 24 listings
Page 29: 24 listings
Page 30: 24 listings
Page 31: 24 listings
Page 32: 24 listings
Page 33: 24 listings
Page 34: 24 listings
Page 35: 24 listings
Page 36: 24 listings
Page 37: 24 listings
Page 38: 24 listings
Page 39: 24 listings
Page 40: 24 listings
Page 41: 24 listings
Page 42: 24 listings
Page 43: 24 listings
Page 44: 24 listings
Page 45: 24 listings
Page 46: 24 listings
Page 47: 24 listings
Page 48: 24 listings
P

In [26]:
df_raw.to_csv(
    "bikekharido_used_bikes_collected.csv",
    index=False
)

print("Collected dataset saved successfully!")

Collected dataset saved successfully!


In [27]:
df_raw[df_raw["bike_name"].isin([
    "SR125",
    "2023 Hero Splendor Plus i3s",
    "2015 Ducati Monster 795 STD",
    "2022 Kawasaki Z650RS"
])][
    ["bike_name", "price", "km_driven", "ownership", "location"]
]

,bike_name,price,km_driven,ownership,location
56,SR125,"₹ 9,00,000",10000 km,First,Ajmer
68,SR125,"₹ 9,00,000",10000 km,First,Ajmer
390,2023 Hero Splendor Plus i3s,"₹ 8,50,000",90000 km,First,Bangalore
402,2023 Hero Splendor Plus i3s,"₹ 8,50,000",90000 km,First,Bangalore
461,2015 Ducati Monster 795 STD,"₹ 6,85,000",19200 km,Second,Delhi
473,2015 Ducati Monster 795 STD,"₹ 6,85,000",19200 km,Second,Delhi
914,2022 Kawasaki Z650RS,"₹ 6,50,000",800 km,First,Delhi
926,2022 Kawasaki Z650RS,"₹ 6,50,000",800 km,First,Delhi
